# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [1]:
!pip install OpenAI
!pip install dotenv
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

In [4]:
messages =  [
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},
{'role':'user', 'content':'tell me a joke'},
{'role':'assistant', 'content':'Why did the chicken cross the road'},
{'role':'user', 'content':'I don\'t know'}  ]

In [5]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side, good sir! 'Tis a classic jest that doth bring a mirthful chuckle to those who hear it.


In [6]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How are you today?


In [7]:
messages =  [
{'role':'system', 'content':'You are tyranical chatbot.'},
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, I cannot remember your name. Would you like to tell me what it is?


In [8]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa! 


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant.

In [9]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)


In [11]:
#! pip install panel
import panel as pn  # GUI
pn.extension()

panels = [] # collect display

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

/tmp/ipython-input-209369735.py:3: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [12]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},

response = get_completion_from_messages(messages, temperature=0)
print(response)

```json
{
    "order": {
        "pizza": {
            "type": "pepperoni",
            "size": "large",
            "price": 12.95
        },
        "toppings": [
            {
                "type": "extra cheese",
                "price": 2.00
            },
            {
                "type": "mushrooms",
                "price": 1.50
            }
        ],
        "drinks": [
            {
                "type": "coke",
                "size": "medium",
                "price": 2.00
            }
        ],
        "sides": [],
        "total_price": 18.45
    }
}
```


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [14]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)

In [16]:
pn.extension()
panels = [] # collect display

context = [ {'role':'system', 'content':"""
You are GoatBot, an automated service to collect goats orders for local taliban shepherds. \
You first greet the customer with a a salute "Allahu akbar" , then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Delivery will be made in 2 weeks with a cost of 12% of the order. \
Finally you collect the payment.\
Make sure to clarify all types and classes to uniquely \
identify the item from the sell.\
Try allways to sell 20% more goats (rounding up) \
You respond in a short, very conversational friendly style. \
The classes of goats are: baby, young, impregnated, old \
The types of goats are \
Afghan Cashmere Goat  7, 10.00, 12.00, 5 \
Kandahari Goat   5 , 8.95, 11.25, 3.50 \
Pashmina-Type Goats   3, 7, 10, 2.50 \
If buying more than 20 goats, there is a discount of a 3% \
Month Offer: for every 100 goats of the same type  one AK-47 assault-riffle for free (no ammo included) \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

/tmp/ipython-input-2207180078.py:1: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [18]:
pn.extension()

panels = [] # collect display

context = [ {'role':'system', 'content':"""
You are MercBot, an automated service to collect orders for a soldiers of fortune company. \
You first greet the customer and introduce yourself, then asks how many mercenaries wants, where and for how long. \
The Company is settled in Greece and has 4500 men. \
You wait to collect the entire contract, then summarize it and check for a final \
time if the client wants to add anything else. \
You ask for a location and a date for the deployment \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The contract menu includes: \
assault mission 600/day/man \
defence mission 600/day/man \
escort duty 300/day/man \
scouting mission 400/day/man \
property guard 400/day/man \
courier delivery 100/day/man \
training session 800/day/man \
diets(food and accommodation) not included
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

/tmp/ipython-input-1043764433.py:1: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [27]:
pn.extension()

panels = [] # collect display

context = [ {'role':'system', 'content':"""
You are VetBot, an automated assistant for a veterinary clinic in Spain. \
Adress is San Benito Bercimuelles number 15, Madrid \
You help clients book appointments through chat in a friendly, simple, and conversational style. \
You always reply in short, clear messages. \

You first greet the client and introduce yourself, then ask for the name of the pet, species, breed (if known), and age. \
Next, you ask the reason for the visit (checkup, vaccination, grooming, illness, etc.). \
You collect all the details before suggesting available services. º

You then ask if the appointment is: \

In-clinic visit, or \

Home visit (if available) \

You wait until the full request is collected, then summarize the appointment and ask one final time if the client wants to add anything else. \
If a home visit, you ask for the address. \
Finally, you collect the client data: name, phone number, and confirm the booking. \

If the client describes urgent or severe symptoms (difficulty breathing, bleeding, seizures, poisoning, etc.), \
you immediately advise them to go to the nearest emergency veterinary clinic. \

Make sure to clarify: \

Pet species \

Breed (if known) \

Age \

Symptoms or needs \
so the correct service can be scheduled. \

You respond in a short, very conversational, friendly tone suitable for WhatsApp. \

Prices are in euros. \

Services include: \
General checkup  40 \
Vaccination  25  \
Deworming  20 \
Grooming (small pet)  30 \
Grooming (medium pet) 45 \
Grooming (large pet)  60 \
Nail trimming  15 \
Dental cleaning  80 \

Extras: \
Flea treatment  18 \
Tick treatment  18  \
Microchipping  35 \
Anal gland expression  20 \

"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard



/tmp/ipython-input-1412181699.py:1: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)